In [5]:
import tensorflow as tf 
from tensorflow import keras 
import tensorflow_addons as tfa 
import pandas as pd
import numpy as np 
from sklearn.metrics import mean_absolute_error
import gensim.downloader as api


### Load text data <br>

In [6]:
train_ds = tf.data.Dataset.load('./data/text/train_ds/').shuffle(buffer_size=1000, seed=42).batch(batch_size=32).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
valid_ds = tf.data.Dataset.load('./data/text/val_ds/').shuffle(buffer_size=1000, seed=42).batch(batch_size=32).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds  = tf.data.Dataset.load('./data/text/test_ds/').shuffle(buffer_size=1000, seed=42).batch(batch_size=32).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

train_ds, valid_ds


(<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 50), dtype=tf.int32, name=None), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>,
 <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 50), dtype=tf.int32, name=None), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>)

### Load embed_matrix

In [7]:
embed_matrix = np.load('./data/text/embed_matrix.npy')

### Build Text model

In [8]:
# vocab_size = 11052 -> dataset penuh
vocab_size = 89
sentlen    = 50
 
inputs = keras.layers.Input(shape=(sentlen))
embed  = keras.layers.Embedding(input_dim=vocab_size, output_dim=100, embeddings_initializer=keras.initializers.Constant(embed_matrix),input_length=sentlen, trainable=False)(inputs)

x = keras.layers.Conv1D(filters=16, kernel_size=3, activation='relu')(embed)
x = keras.layers.Conv1D(filters=8, kernel_size=3, activation='relu')(x)
x = keras.layers.Flatten()(x)
x = keras.layers.Dense(50, activation='relu')(x)

y = keras.layers.Conv1D(filters=32, kernel_size=3, activation='relu')(embed)
y = keras.layers.Conv1D(filters=16, kernel_size=3, activation='relu')(y)
y = keras.layers.Flatten()(y)
y = keras.layers.Dense(50, activation='relu')(y)

z = keras.layers.Concatenate()([x,y])

z = keras.layers.Dense(256, activation='relu')(z)
z = keras.layers.Dense(5, activation='sigmoid')(z)


text_model = keras.models.Model(inputs=inputs, outputs=z, name='text_model')
text_model.compile(loss='mse', optimizer=tfa.optimizers.RectifiedAdam(), metrics=['mae'])


In [9]:
print("Embed Matrix Shape:", embed_matrix.shape)
print("Vocab Size:", vocab_size)


Embed Matrix Shape: (89, 100)
Vocab Size: 89


### Compile & Train model

In [10]:
import datetime
t = datetime.datetime.now().strftime("%m%d_%H%M%S")

early_stopping = keras.callbacks.EarlyStopping(patience=10, verbose=0)
check_point    = keras.callbacks.ModelCheckpoint(filepath='./weights/text/text.t5',
                             monitor='val_mae',
                             mode='min',
                             save_best_only=True,
                             save_weights_only=True,
                             verbose=0)

optimizer = tfa.optimizers.RectifiedAdam()

history = text_model.fit(train_ds, validation_data=valid_ds, batch_size=32, epochs=100, callbacks=[early_stopping, check_point])

Epoch 1/100
1/1 [==============================] - 2s 2s/step - loss: 0.0260 - mae: 0.1507 - val_loss: 0.0113 - val_mae: 0.0863
Epoch 2/100
1/1 [==============================] - 0s 82ms/step - loss: 0.0260 - mae: 0.1506 - val_loss: 0.0112 - val_mae: 0.0862
Epoch 3/100
1/1 [==============================] - 0s 83ms/step - loss: 0.0260 - mae: 0.1505 - val_loss: 0.0112 - val_mae: 0.0862
Epoch 4/100
1/1 [==============================] - 0s 83ms/step - loss: 0.0259 - mae: 0.1504 - val_loss: 0.0112 - val_mae: 0.0861
Epoch 5/100
1/1 [==============================] - 0s 81ms/step - loss: 0.0259 - mae: 0.1503 - val_loss: 0.0112 - val_mae: 0.0860
Epoch 6/100
1/1 [==============================] - 0s 85ms/step - loss: 0.0259 - mae: 0.1502 - val_loss: 0.0110 - val_mae: 0.0850
Epoch 7/100
1/1 [==============================] - 0s 81ms/step - loss: 0.0254 - mae: 0.1488 - val_loss: 0.0108 - val_mae: 0.0838
Epoch 8/100
1/1 [==============================] - 0s 98ms/step - loss: 0.0247 - mae: 0.1469

### Load weights

In [11]:
text_model.load_weights('./weights/text/text.t5')

## Evaluation

### Validation data

In [12]:

valid_ds = tf.data.experimental.load('./data/text/val_ds/').batch(batch_size=32).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds  = tf.data.experimental.load('./data/text/test_ds/').batch(batch_size=32).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

valid_ds, test_ds

Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.


(<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 50), dtype=tf.int32, name=None), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>,
 <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 50), dtype=tf.int32, name=None), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>)

In [13]:
y_true = np.concatenate([y for x,y in valid_ds], axis=0)
y_pred = text_model.predict(valid_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 0s 107ms/step


(array([93.62605 , 98.9777  , 99.78072 , 96.183395, 88.31009 ],
       dtype=float32),
 95.37559263408184)

In [14]:
loss, mae = text_model.evaluate(valid_ds)
(1-mae)*100

1/1 [==============================] - 0s 20ms/step - loss: 0.0039 - mae: 0.0462


95.37559263408184

### Test data

In [15]:
y_true = np.concatenate([y for x,y in test_ds], axis=0)
y_pred = text_model.predict(test_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 0s 18ms/step


(array([88.48927, 84.35418, 94.60074, 97.84063, 99.30084], dtype=float32),
 92.91713386774063)

In [16]:
loss, mae = text_model.evaluate(test_ds)
(1-mae)*100

1/1 [==============================] - 0s 11ms/step - loss: 0.0082 - mae: 0.0708


92.91713386774063